# 17 — Linear probing the encoder roster

Notebooks 11–16 answer *how good is this backbone after we fine-tune it on our labels*. This one
answers the question that comes before it: **how much of that was already there.**

The backbone is frozen and never receives a gradient. One forward pass produces a fixed sentence
vector per row; a logistic regression on those vectors is the only thing that trains. The number
that matters is not the probe's absolute score — it is the **gap to the fine-tune of the same
model on the same task and split**.

Why that gap is worth measuring here specifically:

- `label_ceiling.csv` puts v5-sentiment-vs-human agreement at **0.5769** [0.40, 0.73] and the
  champion fine-tune at **0.566** — inside the ceiling's CI. If the frozen probe lands near the
  fine-tune, fine-tuning is mostly fitting label noise and the next move is a **relabel**, not a
  bigger model. If the probe collapses, the model is genuinely learning task structure.
- `best_epoch` already hints the same way: sentiment peaks at **epoch 1 of 3** then degrades,
  while priority is still improving at **3 of 3**. Priority is the clean-label control in this
  table; sentiment is the suspect.
- **Intent had zero encoder runs of any kind.** The probe is the cheapest thing that says anything
  about it, because extraction is paid once per backbone and reused across all three tasks.

Everything below runs off `swiftbench.probe`. Read its module docstring before changing anything —
the three invariants it enforces (per-model pooling, row L2-normalisation, id-asserted caching)
each exist because the alternative fails silently.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import numpy as np, pandas as pd
import swiftbench as sb
from swiftbench import config, probe, results, splits
from swiftbench.train_encoder import MONOLINGUAL, device

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

AUTHOR = "sithija"
print("split sha:", splits.sha(), "| device:", device())
print("roster   :", probe.ROSTER)
print("poolings : encoder", probe.ENCODER_POOLINGS, "| decoder", probe.DECODER_POOLINGS)

split sha: e7b5934392cd | device: mps
roster   : ['sinhalaberto', 'labse', 'xlmr-base', 'twhin-bert', 'mmbert', 'canine-c', 'sinbert-large', 'gemma-3-270m', 'gemma-3-1b']
poolings : encoder ('cls', 'mean') | decoder ('last', 'mean')


## Two traps this notebook already fell into

Both are recorded here rather than quietly fixed, because both produced *plausible numbers* and
neither raised anything.

**1. A unit-norm feature matrix does not fit at `C=1`.** Row L2-normalisation is required for the
cross-model column to mean anything — Gemma's activation scale is nowhere near BERT's, and without
it a fixed `C` regularises each backbone differently. But a unit-norm row spreads its length across
`dim` coordinates, so every feature is O(1/√768) and the L2 penalty dominates the likelihood
immediately. lbfgs hit its gradient tolerance after **4 iterations** and returned an underfit model.
The intent probe read **0.167** macro-F1. With a `StandardScaler` in front of the regression —
restoring unit per-dimension variance, and incidentally stripping the dominant mean direction that
makes raw transformer embeddings anisotropic — the same backbone, the same vectors, and the same
`C` read **0.807**. `probe.fit()` now reports `n_iter` for exactly this reason.

**2. The fine-tuned checkpoints cannot be scored on dev.** `ml/models/README.md` records that both
LaBSE checkpoints were fit on **`train+dev`** — they were the chosen winners being prepared for a
single test scoring. So dev is inside their training data. Probed on dev, `labse-ft-priority` read
**0.9645** against its own fine-tune's 0.9168, and `labse-ft-sentiment` **0.6498** against 0.633 —
both look like "fine-tuning produced a beautifully linearly-separable representation" and both are
memorised rows. `probe.score()` now refuses the combination outright; the fine-tuned checkpoints
are evaluated on **test** further down, against those checkpoints' recorded *test* numbers.

## Fine-tune numbers to subtract from

Pooled dev, from `ENCODER_FINDINGS.md`. The classical champion is here as the
do-we-even-need-a-transformer line.

In [2]:
# Read from the run files rather than transcribed, so this cannot drift from reports/runs/.
enc = results.load_all("dev")
enc = enc[(enc.family == "encoder") & (enc.eval_lang == "all")]
FINETUNE_DEV = enc.pivot_table(index="model", columns="task", values="headline")
display(FINETUNE_DEV)

CLASSICAL_DEV  = {"sentiment": 0.5950, "priority": 0.9028}                  # tfidf-svm, multi
CLASSICAL_TEST = {"sentiment": 0.4572, "priority": 0.8722, "intent": 0.8318}
CEILING        = {"sentiment": 0.5769, "priority": 0.7722}                  # v5 vs human, gold 500

print("classical dev  :", CLASSICAL_DEV)
print("classical test :", CLASSICAL_TEST)
print("label ceiling  :", CEILING, "  <- priority already sits above its own ceiling")

task,intent,priority,sentiment
model,,,
canine-c,NaN,0.8786,0.5323
gemma-3-1b,0.9243,0.9165,0.6428
gemma-3-270m,0.9038,0.9040,0.5611
labse,0.9224,0.9168,0.6334
mmbert,0.9280,0.9148,0.6203
sinbert-large,NaN,0.4939,0.1238
sinhalaberto,NaN,0.5573,0.1730
twhin-bert,NaN,0.8907,0.5978
xlmr-base,NaN,0.9162,0.5012


classical dev  : {'sentiment': 0.595, 'priority': 0.9028}
classical test : {'sentiment': 0.4572, 'priority': 0.8722, 'intent': 0.8318}
label ceiling  : {'sentiment': 0.5769, 'priority': 0.7722}   <- priority already sits above its own ceiling


## Extraction

Forward-only, cached to `ml/cache/embeddings/` (gitignored, ~2 GB for the full roster). Every
`.npz` carries the `id` array and split sha it was built from, and `probe.features()` asserts both
against the frame it is used with — a stale cache would match labels to the wrong rows and leave
every metric afterwards looking fine.

**Train and dev only for the roster.** Test is not extracted here on purpose: this sweep is model
selection. The only things scored on test are the fine-tuned checkpoints, which have no other
uncontaminated portion.

Re-running is free once the cache is warm. `SMOKE = True` does one small backbone on one language,
so a broken environment fails in seconds rather than an hour.

In [3]:
SMOKE = True          # <- set False for the real extraction

roster = ["sinhalaberto"] if SMOKE else probe.ROSTER
portions = ["dev"] if SMOKE else ["dev", "train"]

for model in roster:
    langs = [MONOLINGUAL[model]] if model in MONOLINGUAL else config.LANGUAGES
    if SMOKE:
        langs = langs[:1]
    for lang in langs:
        for portion in portions:
            probe.embed(model, lang, portion, batch_size=64)
    print(f"== {model} cached")

== sinhalaberto cached


## Does `C = 1.0` cost us anything?

`C=1.0` is the default rather than a swept value: a CV grid over 42,500 × 768 for the 77-way intent
task, nine backbones over, costs more than the extraction does. So it is checked once, here, rather
than assumed. If the curve is flat across two orders of magnitude the default is fine for the whole
roster; if it is not, every cell below needs its own sweep and the table is provisional.

In [4]:
c_path = config.REPORTS_DIR / "probe_C_sweep.csv"
if c_path.exists():
    c_curve = pd.read_csv(c_path)
else:
    c_curve = pd.concat([probe.sweep_C(task=t, model="labse", pooling="mean",
                                       grid=[0.001, 0.01, 0.1, 1.0, 10.0]).assign(task=t)
                         for t in ["sentiment", "priority", "intent"]], ignore_index=True)
    c_curve.to_csv(c_path, index=False)

display(c_curve.pivot_table(index="C", columns="task", values="headline"))
for task, g in c_curve.groupby("task"):
    at1 = g.set_index("C").loc[1.0, "headline"]
    print(f"{task:9s} best C={g.loc[g.headline.idxmax(), 'C']:<7g} "
          f"{g.headline.max():.4f} | C=1.0 {at1:.4f} | cost {g.headline.max() - at1:+.4f}")

task,intent,priority,sentiment
C,,,
0.0010,0.8503,0.8067,0.4047
0.0100,0.8787,0.8266,0.4448
0.1000,0.8756,0.8295,0.4561
1.0000,0.8614,0.8247,0.4563
10.0000,0.8490,0.8243,0.4530


intent    best C=0.01    0.8787 | C=1.0 0.8614 | cost +0.0174
priority  best C=0.1     0.8295 | C=1.0 0.8247 | cost +0.0047
sentiment best C=1       0.4563 | C=1.0 0.4563 | cost +0.0000


## The sweep

Every backbone × every pooling it supports, scored pooled (`eval_lang="all"`) and per language.
`arm="class_weight"` matches the fine-tune roster — a probe scored under a different balancing arm
is not a probe-vs-fine-tune comparison. The fit happens once per (backbone, pooling) and is reused
across all six evaluation cells.

Each result lands in `ml/reports/runs/` under `family: "probe"` and a model name of
`<backbone>-probe-<pooling>`, so probes and fine-tunes share a leaderboard without overwriting
each other.

In [5]:
dev_path = config.REPORTS_DIR / "probe_dev.csv"
if dev_path.exists():
    probe_dev = pd.read_csv(dev_path)
else:
    cached = sorted({p.name.split("__")[0] for p in probe.CACHE_DIR.glob("*.npz")})
    models = [m for m in probe.ROSTER if m in cached]
    probe_dev = pd.concat([probe.sweep(task=t, models=models, author=AUTHOR).assign(task=t)
                           for t in ["sentiment", "priority", "intent"]], ignore_index=True)
    probe_dev.to_csv(dev_path, index=False)

print(f"{len(probe_dev)} cells | backbones: {sorted(probe_dev.model.unique())}")
probe_dev.head()

264 cells | backbones: ['canine-c', 'gemma-3-1b', 'gemma-3-270m', 'labse', 'mmbert', 'sinbert-large', 'sinhalaberto', 'twhin-bert', 'xlmr-base']


,model,pooling,eval_lang,accuracy,macro_f1,macro_precision,macro_recall,weighted_f1,n,negative_f1,negative_precision,negative_recall,n_negative_true,n_negative_pred,headline_metric,headline,n_train,n_train_before_resample,dim,C,backbone,hf_name,fit_portion,fit_seconds,n_iter,frozen,task,f1_low,f1_medium,f1_high
0,sinhalaberto,cls,sinhala,0.9252,0.6914,0.6529,0.7718,0.9356,1498,0.4227,0.3254,0.6029,68.0000,126.0000,negative_f1,0.4227,8500,8500,768,1.0000,sinhalaberto,keshan/SinhalaBERTo,train,0.3000,229,True,sentiment,NaN,NaN,NaN
1,sinhalaberto,mean,sinhala,0.9292,0.7134,0.6689,0.8089,0.9395,1498,0.4646,0.3538,0.6765,68.0000,130.0000,negative_f1,0.4646,8500,8500,768,1.0000,sinhalaberto,keshan/SinhalaBERTo,train,0.2000,237,True,sentiment,NaN,NaN,NaN
2,labse,cls,all,0.9148,0.7101,0.6584,0.8699,0.9316,7490,0.4666,0.3259,0.8206,340.0000,856.0000,negative_f1,0.4666,42500,42500,768,1.0000,labse,sentence-transformers/LaBSE,train,1.6000,333,True,sentiment,NaN,NaN,NaN
3,labse,cls,english,0.9379,0.7565,0.7000,0.8834,0.9476,1498,0.5463,0.4088,0.8235,68.0000,137.0000,negative_f1,0.5463,42500,42500,768,1.0000,labse,sentence-transformers/LaBSE,train,1.6000,333,True,sentiment,NaN,NaN,NaN
4,labse,cls,sinhala,0.9346,0.7513,0.6939,0.8887,0.9454,1498,0.5377,0.3958,0.8382,68.0000,144.0000,negative_f1,0.5377,42500,42500,768,1.0000,labse,sentence-transformers/LaBSE,train,1.6000,333,True,sentiment,NaN,NaN,NaN


## Which pooling won, per backbone

Selected on dev, which is the portion selection is allowed to happen on. Both columns are shown
rather than just the winner: a backbone whose CLS and mean are far apart is telling you where it
stores sentence meaning, and that is worth reading even when it does not change the ranking.

In [6]:
POOL_COLS = ["cls", "mean", "last"]
pooled = probe_dev[probe_dev.eval_lang == "all"]

by_pooling = pooled.pivot_table(index=["task", "model"], columns="pooling",
                                values="headline").reset_index()
present = [c for c in POOL_COLS if c in by_pooling.columns]
by_pooling["best"] = by_pooling[present].max(axis=1)
by_pooling["spread"] = by_pooling["best"] - by_pooling[present].min(axis=1)
display(by_pooling.sort_values(["task", "best"], ascending=[True, False]))

pooling,task,model,cls,last,mean,best,spread
3,intent,labse,0.8590,NaN,0.8614,0.8614,0.0024
1,intent,gemma-3-1b,NaN,0.8408,0.8570,0.8570,0.0162
5,intent,twhin-bert,0.7770,NaN,0.8249,0.8249,0.0480
4,intent,mmbert,0.8054,NaN,0.8170,0.8170,0.0116
6,intent,xlmr-base,0.7906,NaN,0.8074,0.8074,0.0167
2,intent,gemma-3-270m,NaN,0.7918,0.7990,0.7990,0.0073
0,intent,canine-c,0.4734,NaN,0.6410,0.6410,0.1676
10,priority,labse,0.8265,NaN,0.8247,0.8265,0.0017
8,priority,gemma-3-1b,NaN,0.7864,0.7887,0.7887,0.0023
11,priority,mmbert,0.7694,NaN,0.7719,0.7719,0.0025


## Probe vs fine-tune — the table this notebook exists for

`gap` is how much fine-tuning added on top of the frozen representation. `retained` is the fraction
of the fine-tuned score the frozen backbone already reaches.

How to read it:

- **`retained` near 1.0** — the signal was already in pretraining; fine-tuning is moving a decision
  boundary, not learning the task. For sentiment that corroborates the label-ceiling reading:
  relabel, don't re-model.
- **`retained` low, `gap` large** — the model genuinely learned task structure from our labels, and
  the labels carry real information the encoder could not have guessed.
- **Priority is the control.** Cleaner labels (κ=0.64 vs sentiment's 0.55) and a fine-tune still
  improving at the last epoch. If sentiment retains much *more* than priority does, that difference
  is about the labels, not about one task being easier.

In [7]:
for task in ["sentiment", "priority", "intent"]:
    d = probe.deltas(task)
    print(f"\n=== {task} — headline = {sb.metrics.HEADLINE[task]}")
    if d.empty:
        best = (pooled[pooled.task == task]
                .sort_values("headline", ascending=False)
                .groupby("model", as_index=False).first()[["model", "pooling", "headline"]])
        print("  no fine-tune recorded at this split sha to subtract from; probes alone:")
        display(best)
    else:
        display(d)


=== sentiment — headline = negative_f1


,backbone,pooling,probe,finetune,gap,retained
0,labse,cls,0.4666,0.6334,0.1669,0.7366
1,gemma-3-1b,last,0.4391,0.6428,0.2036,0.6832
2,mmbert,mean,0.4137,0.6203,0.2065,0.6671
3,xlmr-base,mean,0.4068,0.5012,0.0944,0.8116
4,twhin-bert,mean,0.3828,0.5978,0.2150,0.6403
5,gemma-3-270m,last,0.3539,0.5611,0.2072,0.6307
6,canine-c,mean,0.3164,0.5323,0.2158,0.5945



=== priority — headline = macro_f1


,backbone,pooling,probe,finetune,gap,retained
0,labse,cls,0.8265,0.9168,0.0904,0.9014
1,gemma-3-1b,mean,0.7887,0.9165,0.1278,0.8606
2,mmbert,mean,0.7719,0.9148,0.1429,0.8438
3,twhin-bert,mean,0.7685,0.8907,0.1222,0.8628
4,xlmr-base,mean,0.7643,0.9162,0.1519,0.8343
5,gemma-3-270m,last,0.7255,0.9040,0.1784,0.8026
6,canine-c,mean,0.6565,0.8786,0.2221,0.7472



=== intent — headline = macro_f1


,backbone,pooling,probe,finetune,gap,retained
0,labse,mean,0.8614,0.9224,0.0610,0.9338
1,gemma-3-1b,mean,0.8570,0.9243,0.0673,0.9272
2,mmbert,mean,0.8170,0.9280,0.1110,0.8804
3,gemma-3-270m,mean,0.7990,0.9038,0.1048,0.8841


## Per-language

The pooled cell hides the thing the project actually cares about — whether the frozen
representation carries signal on the romanized tracks, where every fine-tune result so far is
suspect because Singlish is rule-generated and Tamilish is machine-translated.

A probe is a *cleaner* instrument there than a fine-tune is: it cannot memorise the generator's
regularities, because nothing in the backbone moves. A backbone that probes well on Singlish is
carrying real cross-lingual transfer.

In [8]:
best_pool = (pooled.sort_values("headline", ascending=False)
                   .groupby(["task", "model"], as_index=False)
                   .first()[["task", "model", "pooling"]])

per_lang = (probe_dev.merge(best_pool, on=["task", "model", "pooling"])
                     .query("eval_lang != 'all'")
                     .pivot_table(index=["task", "model"], columns="eval_lang", values="headline"))
per_lang = per_lang[[l for l in config.LANGUAGES if l in per_lang.columns]]
per_lang["native_minus_roman"] = (
    per_lang[["sinhala", "tamil"]].mean(axis=1) - per_lang[["singlish", "tamilish"]].mean(axis=1)
)
display(per_lang)

eval_lang               english  sinhala  singlish  tamil  tamilish  native_minus_roman
task      model                                                                        
intent    canine-c       0.5910   0.6748    0.6848 0.6087    0.6421             -0.0217
          gemma-3-1b     0.8557   0.8762    0.8362 0.8892    0.8278              0.0507
          gemma-3-270m   0.7743   0.8199    0.8047 0.8410    0.7535              0.0514
          labse          0.8780   0.8883    0.8507 0.8533    0.8368              0.0271
          mmbert         0.8481   0.8455    0.7790 0.8590    0.7514              0.0870
          twhin-bert     0.8251   0.8482    0.8278 0.8042    0.8194              0.0026
          xlmr-base      0.8359   0.8427    0.7803 0.8338    0.7432              0.0765
priority  canine-c       0.6488   0.6780    0.6646 0.6384    0.6529             -0.0006
          gemma-3-1b     0.8252   0.7985    0.7935 0.7822    0.7453              0.0209
          gemma-3-270m   0.8013   0.7069    0.7517 0.6687    0.7041             -0.0401
          labse          0.8576   0.8678    0.7953 0.8462    0.7695              0.0746
          mmbert         0.8233   0.7821    0.7441 0.7947    0.7195              0.0566
          twhin-bert     0.7891   0.7803    0.7686 0.7589    0.7463              0.0122
          xlmr-base      0.7945   0.8020    0.7367 0.7776    0.7141              0.0644
sentiment canine-c       0.3174   0.3259    0.3312 0.2866    0.3228             -0.0208
          gemma-3-1b     0.4500   0.4324    0.4375 0.4552    0.4226              0.0138
          gemma-3-270m   0.4094   0.3478    0.3663 0.3193    0.3438             -0.0214
          labse          0.5463   0.5377    0.4000 0.4907    0.4028              0.1128
          mmbert         0.5673   0.4055    0.3849 0.4014    0.3545              0.0338
          twhin-bert     0.4167   0.3984    0.3612 0.3849    0.3579              0.0321
          xlmr-base      0.4800   0.4615    0.3793 0.4075    0.3279              0.0810

## What did fine-tuning put into the representation?

`labse-ft-sentiment` and `labse-ft-priority` are our own fine-tuned checkpoints from
`ml/models/encoders/`, probed the same way. The classification head is discarded —
`AutoModel.from_pretrained` loads the backbone only — so this isolates what fine-tuning changed
about the *representation*, separately from what it put in the head.

**Scored on test**, because both were fit on `train+dev` and test is the only portion their
backbones never saw. Compare against those checkpoints' recorded test numbers, not their dev ones:
sentiment **0.5664**, priority **0.8900**.

Three readings:

- **ft-probe ≈ pretrained-probe** — fine-tuning barely moved the representation; whatever it gained
  lives in the head.
- **ft-probe ≫ pretrained-probe** — the backbone genuinely reorganised around the task.
- **ft-probe on the *other* task ≈ pretrained** — the reorganisation is task-specific and does not
  transfer, which is the argument against one shared encoder with three heads.

That last row is why both checkpoints are probed on all three tasks rather than each on its own.

In [9]:
FT_TEST = {"sentiment": 0.5664, "priority": 0.8900}   # the checkpoints' own recorded test scores

ft_path = config.REPORTS_DIR / "probe_test_finetuned.csv"
if ft_path.exists():
    ft = pd.read_csv(ft_path)
    wide = ft.pivot_table(index=["task", "pooling"], columns="checkpoint", values="headline")
    display(wide)
    if "labse" in wide.columns:
        for tag in [c for c in wide.columns if c.startswith("labse-ft")]:
            print(f"{tag:22s} minus pretrained labse:")
            print((wide[tag] - wide["labse"]).round(4).to_string())
else:
    print("run the fine-tuned-checkpoint pass first (see ml/README.md)")

checkpoint         labse  labse-ft-priority  labse-ft-sentiment
task      pooling                                              
intent    cls     0.7773             0.7962              0.7594
          mean    0.7822             0.7971              0.7676
priority  cls     0.8008             0.8825              0.7990
          mean    0.8015             0.8816              0.8040
sentiment cls     0.3682             0.3828              0.4633
          mean    0.3735             0.3867              0.4810

labse-ft-priority      minus pretrained labse:
task       pooling
intent     cls       0.0189
           mean      0.0150
priority   cls       0.0817
           mean      0.0801
sentiment  cls       0.0146
           mean      0.0132
labse-ft-sentiment     minus pretrained labse:
task       pooling
intent     cls       -0.0179
           mean      -0.0146
priority   cls       -0.0018
           mean       0.0025
sentiment  cls        0.0951
           mean       0.1075


## What this run concluded

Written up in full as §6 of `ml/reports/ENCODER_FINDINGS.md`. The four results:

**1. Retention is ordered intent > priority > sentiment, for every backbone, with no exceptions.**
Intent is 88–93% linearly decodable from a representation that never saw our data; sentiment is
59–74%. Fine-tuning does the most work on exactly the task whose labels agree with humans least
(κ=0.55 against priority's 0.64) — consistent with the label-ceiling reading, and it says the
sentiment gap is *not* something a better frozen encoder closes.

**2. LaBSE wins the frozen probe on all three tasks and retains the most on all three.** The
fine-tune bake-off's champion was already the champion before any training; its advantage lives in
the pretrained representation, not in how it responds to our labels.

**3. What fine-tuning adds to the representation is task-specific and does not transfer.** On test,
`labse-ft-sentiment` gives +0.108 on sentiment and +0.003 on priority; `labse-ft-priority` gives
+0.081 on priority and +0.013 on sentiment. The diagonal is everything. → **a shared backbone with
three heads will not work**, which settles an option that was open in `ml/models/README.md`.

**4. A frozen backbone is not servable here.** Pretrained LaBSE probed on test reaches priority
0.8015 and intent 0.7822, both *below* the classical TF-IDF champion (0.8722, 0.8318). The cheap
serving idea — one frozen encoder, three logistic regressions — is measured and rejected.

One side finding worth carrying into the romanized work: **TwHIN-BERT's representation is the most
script-agnostic in the roster** (native-minus-romanized gap 0.003–0.032, against LaBSE's
0.027–0.113), which is what `model-research.md` §5 predicted and what the fine-tune comparison in
§3 could not see. It still loses on absolute quality, so the ship decision is unchanged — but the
reason it loses is quality, not script handling.